In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv("smartcart_customers.csv")
df

In [ ]:
df.isnull().sum()

# Data prepocessing 

## 1. Handle missing values

In [ ]:
df["Income"]=df["Income"].fillna(df["Income"].median())

In [ ]:
df.isnull().sum()

## 2.Feature Engineering

In [ ]:
df["Age"]=2026-df["Year_Birth"]

In [ ]:
df["Dt_Customer"]=pd.to_datetime(df["Dt_Customer"],dayfirst=True)

reference_date=df["Dt_Customer"].max()

df["Customer_Tenure_Days"]=(reference_date-df["Dt_Customer"]).dt.days

In [ ]:
df["Total_Spending"]=df["MntWines"]+df["MntFruits"]+df["MntMeatProducts"]+df["MntFishProducts"]+df["MntSweetProducts"]+df["MntGoldProds"]

In [ ]:
df["Total_Children"]=df["Teenhome"]+df["Kidhome"]

In [ ]:
df["Education"].value_counts()

df["Education"]=df["Education"].replace(
    {
        "Graduation":"Graduate",
        "PhD":"Postgraduate",
        "Master":"Postgraduate",
        "2n Cycle":"Undergraduate",
        "Basic":"Undergraduate"
    }
)

In [ ]:
df["Education"].value_counts()

In [ ]:
df["Living_With"]=df["Marital_Status"].replace(
    {
        "Married":"Partner",
        "Together":"Partner",
        "Single":"Alone",
        "Widow":"Alone",
        "Divorced":"Alone",
        "Absurd":"Alone",
        "YOLO":"Alone",
        
    }
)

In [ ]:
df["Living_With"].value_counts()

In [ ]:
df.head()

## Drop Columns

In [ ]:
cols=["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
spending_cols=["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]

cols_to_drop=cols+spending_cols

df_cleaned=df.drop(columns=cols_to_drop)

In [ ]:
df_cleaned.shape

In [ ]:
cols=["Income","Recency","Response","Age","Total_Spending","Total_Children"]

sns.pairplot(df_cleaned[cols])

In [ ]:
#Remove outliers



df_cleaned = df_cleaned[(df_cleaned["Age"]<90)]
df_cleaned = df_cleaned[(df_cleaned["Income"]<600_000)]

print("data size without outliers:", len(df_cleaned))

## Heatmap

In [ ]:
corr=df_cleaned.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    corr,
    annot=True,
    annot_kws={"size":6},
    cmap="coolwarm"
)

## Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder


In [ ]:
ohe=OneHotEncoder()

cat_cols=["Education","Living_With"]

enc_cols=ohe.fit_transform(df_cleaned[cat_cols])

In [ ]:
enc_df=pd.DataFrame(enc_cols.toarray(),columns=ohe.get_feature_names_out(cat_cols),index=df_cleaned.index)

In [ ]:
df_encoded=pd.concat([df_cleaned.drop(columns=cat_cols),enc_df],axis=1)

## Scaling

In [ ]:
X=df_encoded

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)


In [ ]:
X_scaled.shape

## Visualize

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca=PCA(n_components=3)

X_pca=pca.fit_transform(X_scaled)


In [ ]:
fig=plt.figure(figsize=(8,6))

ax=fig.add_subplot(111,projection="3d")

ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2])

ax.set_xlabel("PCA1")
ax.set_ylabel("PCA2")
ax.set_zlabel("PCA3")
ax.set_title("3d projection")

In [ ]:
pca.explained_variance_ratio_

# Analyze K Value
## 1.Elbow Method

In [ ]:
from sklearn.cluster import KMeans
wcss=[]
for k in range(1,11):
    kmeans=KMeans(n_clusters=k,random_state=42)
    kmeans.fit_predict(X_pca)
    wcss.append(kmeans.inertia_)
    

In [ ]:


from kneed import KneeLocator

k=KneeLocator(range(1,11),wcss,curve="convex",direction="decreasing")
optimal_k=k.elbow

In [ ]:
print("best k val:",optimal_k)

In [ ]:
#plot
plt.plot(range(1,11),wcss,marker="o")
plt.xlabel("K")
plt.ylabel("WCSS")

## 2.Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score
ss=[]
for k in range(2,11):
    kmeans=KMeans(n_clusters=k,random_state=42)
    labels=kmeans.fit_predict(X_pca)
    ss.append(silhouette_score(X_pca,labels))
    

In [ ]:
plt.plot(range(2,11),ss,marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette Score")

In [ ]:
#combined plot
K_range=range(2,11)

figure,ax1=plt.subplots(figsize=(8,6))

ax1.plot(K_range,wcss[:len(K_range)],marker="o",color="blue")
ax1.set_xlabel("K")
ax1.set_ylabel("WCSS")

ax2=ax1.twinx()
ax2.plot(K_range,ss[:len(K_range)],marker="x",color="red",linestyle="--")
ax2.set_ylabel("ss")
#as we can se the both cut at k=4 so 4 clusters should be taken generally ss high is good but we need to check for both if elbow and silhouette both taken into consideration because as silhouette score increases wcss decreases

## Clustering 

In [ ]:

kmeans=KMeans(n_clusters=4,random_state=42)
labels_kmeans=kmeans.fit_predict(X_pca)

fig=plt.figure(figsize=(8,6))

ax=fig.add_subplot(111,projection="3d")

ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2],c=labels_kmeans)



In [ ]:
#Agglomerative Clustering

from sklearn.cluster import AgglomerativeClustering

agg_clus=AgglomerativeClustering(n_clusters=4,linkage="ward")
labels_agg=agg_clus.fit_predict(X_pca)

fig=plt.figure(figsize=(8,6))

ax=fig.add_subplot(111,projection="3d")

ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2],c=labels_agg)

# Characterization of Clusters

In [ ]:
X["cluster"]=labels_agg

In [ ]:
X.head()

In [ ]:
pal=["red","blue","yellow","green"]

sns.countplot(x=X["cluster"],palette=pal,hue=X["cluster"])

In [ ]:
sns.scatterplot(x=X["Total_Spending"],y=X["Income"],hue=X["cluster"],palette=pal)

In [ ]:
#Insight :- 1. C0-low-mod income/low-mod spend   2.c1:-high inc/high spend 3.c2:-low inc/low spend 4.c3:-mod-high income/high spend


# Cluster Summary

In [ ]:
cluster_summary=X.groupby("cluster").mean()

In [ ]:
print(cluster_summary)